# Parameter Sensitivity Analysis

Loads `experiments/results.csv` and tests whether observed differences across
parameter levels are statistically significant.

**Sweeps**
- **A** — generations (5, 10, 20, 40), `population_size=200`
- **B** — population size (50, 100, 200, 500, 1000), `generations=10`
- **C** — fitness weight presets

Point the `RESULTS_CSV` env var at another file to read a different copy — an
archived or per-machine CSV, say (works for `jupyter nbconvert --execute` too).

**Statistical tests**
- `best_fitness` (continuous): Kruskal-Wallis H; post-hoc Dunn with Holm correction
- `implies_ideal` (binary, 1 = repair is equivalent or stronger than an ideal):
  chi-square; post-hoc Fisher's exact with Bonferroni correction


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp
from pathlib import Path

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
sns.set_palette('tab10')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.4f}'.format)


In [ ]:
import os

results_path = Path(os.environ.get('RESULTS_CSV', '../experiments/results.csv'))
if not results_path.exists():
    raise FileNotFoundError(
        f'Results not found: {results_path}  --  run: python scripts/run_experiments.py'
    )

df = pd.read_csv(results_path)
# Coerce level_value to numeric where possible (A/B are integers, D-I fractional)
df['level_value'] = pd.to_numeric(df['level_value'], errors='coerce').fillna(df['level_value'])

# A CSV predating the selection column holds nsga2 runs only (every config in
# use pinned it). Defaulting keeps the rest of the notebook uniform.
if 'selection' not in df.columns:
    df['selection'] = 'nsga2'
df['selection'] = df['selection'].fillna('nsga2')

SCHEMES = sorted(df['selection'].unique())

print(f'Loaded {len(df)} rows from {results_path.name}')
print(f'Selection schemes: {", ".join(SCHEMES)}')
print(df.groupby(['sweep', 'selection'])['seed'].count().rename('n_rows').to_string())
df.head()


In [ ]:
def analyse_sweep(df, sweep, xlabel, sort_numeric=True):
    sdf = df[df['sweep'] == sweep].copy()
    if sdf.empty:
        print(f'No data for sweep {sweep}')
        return

    # Determine display order for levels
    lv = sdf.drop_duplicates('level_name')[['level_name', 'level_value']].copy()
    if sort_numeric:
        lv['_n'] = pd.to_numeric(lv['level_value'], errors='coerce')
        lv = lv.sort_values('_n')
    order = lv['level_name'].tolist()

    # Selection scheme is a factor, not a nuisance: pooling nsga2 and weighted
    # into one distribution per level would compare the wrong things.
    for scheme in sorted(sdf['selection'].unique()):
        scheme_df = sdf[sdf['selection'] == scheme]
        for spec in sorted(scheme_df['spec'].unique()):
            spec_df = scheme_df[scheme_df['spec'] == spec].copy()
            if spec_df.empty:
                continue

            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            fig.suptitle(f'Sweep {sweep} - {spec} - {scheme}  ({xlabel})', fontsize=13)

            # Panel 1: best_fitness distribution
            valid = spec_df[spec_df['best_fitness'].notna()]
            if not valid.empty:
                sns.boxplot(data=valid, x='level_name', y='best_fitness', order=order, ax=axes[0])
            axes[0].set(title='Best fitness score', xlabel=xlabel, ylabel='Fitness')
            axes[0].tick_params(axis='x', rotation=45)

            # Panel 2: implies_ideal rate
            rates = spec_df.groupby('level_name')['implies_ideal'].mean().reindex(order).fillna(0)
            axes[1].bar(range(len(order)), rates.values, color='steelblue')
            axes[1].set_xticks(range(len(order)))
            axes[1].set_xticklabels(order, rotation=45)
            axes[1].set(title='Rate: repair implies ideal', xlabel=xlabel, ylabel='Rate', ylim=(0, 1.05))

            # Panel 3: found_repair rate
            found = spec_df.groupby('level_name')['found_repair'].mean().reindex(order).fillna(0)
            axes[2].bar(range(len(order)), found.values, color='darkorange')
            axes[2].set_xticks(range(len(order)))
            axes[2].set_xticklabels(order, rotation=45)
            axes[2].set(title='Rate: any repair found', xlabel=xlabel, ylabel='Rate', ylim=(0, 1.05))

            plt.tight_layout()
            plt.show()

            print()
            print(f'=== Sweep {sweep} / {spec} / {scheme} ===')

            # --- Kruskal-Wallis on best_fitness ---
            fit_groups = [
                spec_df[spec_df['level_name'] == ln]['best_fitness'].dropna().values
                for ln in order
            ]
            fit_groups = [g for g in fit_groups if len(g) > 0]
            # stats.kruskal raises when every observation is identical. That happens
            # when a sweep is saturated — the GA converges to the same optimum at
            # every level — which is a result worth printing, not an error.
            all_fit = np.concatenate(fit_groups) if fit_groups else np.array([])
            if len(fit_groups) >= 2 and all_fit.size and np.ptp(all_fit) == 0:
                print(f'  Kruskal-Wallis (best_fitness):  not applicable - every run '
                      f'scored {all_fit[0]:.6g}, so the levels cannot be separated')
            elif len(fit_groups) >= 2:
                H, p_kw = stats.kruskal(*fit_groups)
                n = int(spec_df['best_fitness'].notna().sum())
                k = len(fit_groups)
                eps2 = (H - k + 1) / (n - k) if n > k else float('nan')
                sig = ' ***' if p_kw < 0.001 else ' **' if p_kw < 0.01 else ' *' if p_kw < 0.05 else ' ns'
                print(f'  Kruskal-Wallis (best_fitness):  H={H:.3f}  p={p_kw:.4f}{sig}  eps2={eps2:.3f}')
                if p_kw < 0.05:
                    valid_df = spec_df[spec_df['best_fitness'].notna()]
                    dunn = sp.posthoc_dunn(
                        valid_df, val_col='best_fitness',
                        group_col='level_name', p_adjust='holm'
                    )
                    present = [l for l in order if l in dunn.index]
                    print('  Post-hoc Dunn (Holm correction):')
                    print(dunn.reindex(index=present, columns=present).round(4).to_string())

            # --- Chi-square on implies_ideal ---
            ct = pd.crosstab(spec_df['level_name'], spec_df['implies_ideal'])
            ct = ct.reindex(order).fillna(0).astype(int)
            # A single outcome column means implies_ideal never varied, so there is
            # nothing to separate. Report that rather than skipping in silence.
            if len(ct.columns) == 1 and len(ct) >= 2:
                print(f'  Chi-square (implies_ideal):     not applicable - every run '
                      f'scored {ct.columns[0]}, so the levels cannot be separated')
            elif not (0 in ct.columns and 1 in ct.columns and len(ct) >= 2):
                print(f'  Chi-square (implies_ideal):     not applicable - '
                      f'{len(ct)} level(s), {len(ct.columns)} outcome(s)')
            if 0 in ct.columns and 1 in ct.columns and len(ct) >= 2:
                chi2_stat, p_chi, dof, _ = stats.chi2_contingency(ct)
                sig = ' ***' if p_chi < 0.001 else ' **' if p_chi < 0.01 else ' *' if p_chi < 0.05 else ' ns'
                print(f'  Chi-square (implies_ideal):     chi2={chi2_stat:.3f}  p={p_chi:.4f}{sig}  dof={dof}')
                if p_chi < 0.05:
                    pairs = []
                    for i, a in enumerate(order):
                        for b in order[i + 1:]:
                            a1 = int(ct.loc[a, 1]) if a in ct.index else 0
                            a0 = int(ct.loc[a, 0]) if a in ct.index else 0
                            b1 = int(ct.loc[b, 1]) if b in ct.index else 0
                            b0 = int(ct.loc[b, 0]) if b in ct.index else 0
                            _, pv = stats.fisher_exact([[a1, a0], [b1, b0]])
                            pairs.append((a, b, pv))
                    n_pairs = len(pairs)
                    for a, b, pv in sorted(pairs, key=lambda x: x[2]):
                        padj = min(pv * n_pairs, 1.0)
                        mark = ' *' if padj < 0.05 else ''
                        print(f'    {a} vs {b}:  p_raw={pv:.4f}  p_adj={padj:.4f}{mark}')
            print()


## Sweep A — Generations

Levels: 5, 10, 20, 40.  `population_size=200`, default fitness weights.


In [ ]:
analyse_sweep(df, 'A', 'Generations', sort_numeric=True)


## Sweep B — Population Size

Levels: 50, 100, 200, 500, 1000.  `generations=10`, default fitness weights.


In [ ]:
analyse_sweep(df, 'B', 'Population Size', sort_numeric=True)


## Sweep C — Fitness Weight Presets

Levels: `default`, `syntactic-heavy`, `semantic-heavy`, `status-only`,
`no-halstead`.  Runs by default; the cell below skips it if the CSV was
narrowed with `--sweeps`.


In [ ]:
if 'C' in df['sweep'].values:
    analyse_sweep(df, 'C', 'Weight Preset', sort_numeric=False)
else:
    print('Sweep C not in results.csv.')
    print('Run: python scripts/run_experiments.py --sweeps C')


## Sweeps D–J — Mutation, Crossover, Bound, Filters

Present only in the `factorial` profile. Each holds everything else at the
baseline, so the default level of each is the same run as `A/gen10`.

- **D/E/F** — the per-operator mutation probabilities `p_trigger`, `p_response`, `p_timing`
- **G** — `default_bound`, the bounded model-counting horizon
- **H** — `crossover_rate`; the `cross0.0` level tests whether crossover contributes at all
- **I** — `mutation_rate`
- **J** — ablation of the weakening filter


In [ ]:
NEW_SWEEPS = [
    ('D', 'p_trigger',      True),
    ('E', 'p_response',     True),
    ('F', 'p_timing',       True),
    ('G', 'default_bound',  True),
    ('H', 'crossover_rate', True),
    ('I', 'mutation_rate',  True),
    ('J', 'run_weakening',  False),
]

present = [s for s, _, _ in NEW_SWEEPS if s in df['sweep'].values]
if not present:
    print('Sweeps D-J not in this CSV — they come from the factorial profile.')
    print('Run: python scripts/run_experiments.py --profile factorial')
else:
    for sweep, xlabel, numeric in NEW_SWEEPS:
        if sweep in df['sweep'].values:
            analyse_sweep(df, sweep, xlabel, sort_numeric=numeric)


## Selection Scheme — NSGA-II vs Weighted

The whole grid is duplicated across both schemes, so the comparison is per
level rather than only at the baseline: it answers whether NSGA-II wins
everywhere or only in a corner of the parameter space.

- `implies_ideal` (binary): Fisher's exact, per spec
- `best_fitness` (continuous): Mann-Whitney U, per spec


In [ ]:
if len(SCHEMES) < 2:
    print(f'Only one selection scheme present ({SCHEMES[0]}) — nothing to compare.')
    print('Run: python scripts/run_experiments.py --profile factorial')
else:
    a, b = SCHEMES[0], SCHEMES[1]

    # Per-spec headline: does the scheme move implies_ideal / best_fitness?
    print(f'{a} vs {b}, pooled over every level\n')
    for spec in sorted(df['spec'].unique()):
        sdf = df[df['spec'] == spec]
        ga, gb = sdf[sdf['selection'] == a], sdf[sdf['selection'] == b]

        a1, a0 = int(ga['implies_ideal'].sum()), int((1 - ga['implies_ideal']).sum())
        b1, b0 = int(gb['implies_ideal'].sum()), int((1 - gb['implies_ideal']).sum())
        line = f'  {spec:14} implies_ideal  {a}: {a1}/{a1+a0} ({a1/max(a1+a0,1):.1%})   {b}: {b1}/{b1+b0} ({b1/max(b1+b0,1):.1%})'
        if min(a1 + a0, b1 + b0) > 0 and (a1 + b1) > 0 and (a0 + b0) > 0:
            _, p = stats.fisher_exact([[a1, a0], [b1, b0]])
            sig = ' ***' if p < 0.001 else ' **' if p < 0.01 else ' *' if p < 0.05 else ' ns'
            line += f'   p={p:.4g}{sig}'
        else:
            line += '   (no variation)'
        print(line)

        fa = ga['best_fitness'].dropna().values
        fb = gb['best_fitness'].dropna().values
        if len(fa) and len(fb) and np.ptp(np.concatenate([fa, fb])) > 0:
            U, p = stats.mannwhitneyu(fa, fb, alternative='two-sided')
            sig = ' ***' if p < 0.001 else ' **' if p < 0.01 else ' *' if p < 0.05 else ' ns'
            print(f'  {"":14} best_fitness   {a}: {np.median(fa):.4f}   {b}: {np.median(fb):.4f}'
                  f'   U={U:.0f}  p={p:.4g}{sig}')
        print()

    # Per-level view: a scheme that only wins in one corner shows up here.
    rate = (df.groupby(['sweep', 'level_name', 'selection'])['implies_ideal']
              .mean().unstack('selection'))
    if a in rate.columns and b in rate.columns:
        rate['delta'] = rate[a] - rate[b]
        print(f'implies_ideal rate by level ({a} - {b}), most negative first:')
        print(rate.sort_values('delta').round(3).to_string())


## Cross-Sweep Summary


In [ ]:
summary = (
    df.groupby(['sweep', 'level_name', 'selection', 'spec'])
    .agg(
        n_runs=('seed', 'count'),
        found_repair_rate=('found_repair', 'mean'),
        implies_ideal_rate=('implies_ideal', 'mean'),
        median_fitness=('best_fitness', 'median'),
        mean_wall_s=('wall_time_s', 'mean'),
    )
    .round(3)
)
summary
